# CADET-GUI: Configuration + Solution widgets

This notebook walks through the widgets built so far in `cadetgui.widgets.composite`:

- **`ConfigurationWidget`** — pick a unit operation (column) and a model-builder template (e.g. LWE), configure both, and build a real CADET-Process process object. Includes Python-script export.
- **`SolutionWidget`** — run that process through the CADET-Process simulator and plot the result, with built-in run history.
- **`ParameterEstimationWidget`** — import a measured chromatogram, calibrate it, and preview it overlaid against a simulated signal.

All are widgets: they render inline in this notebook, and the CADET-Process objects they build stay fully accessible afterwards for scripting — the GUI never traps you inside it.

In [ ]:
from cadetgui.widgets.composite import (
    ConfigurationWidget,
    ParameterEstimationWidget,
    SolutionWidget,
)

## 1. Configure a process

Displaying `ConfigurationWidget` gives you three things to pick: the number of components, a column/unit operation type, and a model-builder template. Changing any of them rebuilds the two forms below it.

1. Pick a column (e.g. `LRMP`) and a model (e.g. `Load–Wash–Elute (LWE)`).
2. Edit the column's or model's fields directly — each change applies automatically as soon as it's valid, no Apply button needed. An invalid value (e.g. a negative length) is flagged inline and shown in that form's status line instead of being applied.

Once every field is valid, `config.process` already holds a real CADET-Process object.

In [ ]:
config = ConfigurationWidget()
config.display()

## 2. Run and visualize the simulation

`SolutionWidget.bind_to_config(config)` makes it track whatever process `config` builds — since fields apply automatically, this widget picks up every valid edit above right away, with no extra step.

Click **Run simulation** below, then use the **Signal** dropdown to plot different unit/port combinations (feed, column outlet, ...). Every run (successful or not) is logged in the **Run:** picker above the signal dropdown — pick an older entry to go back and view it; a failed run shows its error instead of a broken plot.

In [ ]:
solution = SolutionWidget()
solution.bind_to_config(config)
solution.display()

## 3. Import experimental data and compare

`ParameterEstimationWidget.bind_to_config(config)` tracks `config`'s built process for fitting. It nests a `DataImportWidget` (`.data`, CSV upload, first two columns read as time/signal) as its experimental-data source. This tab owns the comparison view — the Solution tab above only ever shows the raw simulation, never an overlay.

1. Click **Preview** to simulate the current configuration once and populate the **Signal** dropdown (defaults to the process outlet — "Sink" — since that's what's usually fit against).
2. Upload one of the three example datasets in `examples/data/`, or your own CSV.
3. Check one or more parameters in the checklist, adjust each one's **Start** value if the config's current value isn't a good starting guess (it's a genuinely separate, editable field — not implied by the configuration), and click **Run estimation**.

See **`examples/EXAMPLE_DATA.md`** for exactly what configuration each example file was generated from (so you know the "correct" answer before fitting) and what to check in the parameter checklist for each:

- `example_uv_signal.csv` — a synthetic UV280 trace requiring a **Calibration** step (Beer-Lambert) before it overlays the simulated curve.
- `example_raw_signal.csv` — the simplest fitting exercise: one parameter (`total_porosity`), no calibration needed.
- `example_binding_signal.csv` — a bind-and-elute scenario: 3-4 parameters (porosity, binding rates, optionally axial dispersion), no calibration needed.

In [ ]:
parameter_estimation = ParameterEstimationWidget()
parameter_estimation.bind_to_config(config)
parameter_estimation.display()

## 4. The underlying objects stay accessible

Nothing above is trapped inside the widgets. `config.process` is a plain CADET-Process object, and `solution.result` is a plain `SimulationResults` object — use them exactly as you would if you'd built everything by hand in a script, for anything the GUI doesn't cover. `solution.history.runs` and `parameter_estimation.data.datasets` are plain lists too.

In [ ]:
print(type(config.process))
print(type(solution.result))
print(list(solution.result.solution.keys()))  # unit operation names
n_datasets = len(parameter_estimation.data.datasets)
print(f"{len(solution.history.runs)} run(s) recorded, {n_datasets} dataset(s) loaded")